### Addressing the Empty `sales_transformed_df`

The `sales_transformed_df` is currently empty, and the `sales_transform_report` indicates that all 1000 rows were rejected. This occurred because the `UnitPrice` column in the raw `sales.csv` is entirely `NaN`.

As per the project guidelines, we should not reject records if there's a possibility to enrich the data later. The `UnitPrice` (which corresponds to `price_usd` in the products data) can be joined from the `products_transformed_df` during a subsequent data integration phase.

Therefore, I will modify the `transform_sales_data` function to:
1.  **Standardize column names to snake_case.**
2.  **Convert `OrderDate` to datetime.**
3.  **Keep `unit_price`, `sales_amount`, and `total_price` as `NaN` for now**, as they will be populated from the `products` table during the `integration` phase.
4.  **Crucially, _not reject rows solely based on `unit_price` being missing_** at this `transform` stage. Rows will only be rejected if critical identifiers like `customer_id`, `product_id`, or `quantity` are missing.

After redefining the function, I will re-execute the sales transformation to correctly populate `sales_transformed_df`.

In [ ]:
import pandas as pd
import numpy as np
import re

def transform_sales_data(df: pd.DataFrame) -> (pd.DataFrame, dict, list):
    """
    Transforms the raw sales data.
    - Standardizes column names.
    - Corrects data types for order_date.
    - Handles missing unit_price by keeping it NaN for later enrichment
      from the products data. No rejection based on missing unit_price.
    - Calculates sales_amount and total_price, which will be NaN if unit_price is NaN.
    - Rejects rows only if critical identifiers (customer_id, product_id, quantity) are missing.
    """
    report = {
        'issues': [],
        'extracted_rows': len(df),
        'transformed_rows': 0,
        'rejected_rows': 0,
        'data_type_errors': [],
        'final_missing_values': 0
    }
    rejected_records = []

    # Create a copy to avoid modifying the original extracted_df
    df_transformed = df.copy()

    # 1. Standardize column names to snake_case using explicit mapping for robustness
    column_rename_map = {
        'OrderID': 'order_id',
        'OrderDate': 'order_date',
        'CustomerID': 'customer_id',
        'ProductID': 'product_id',
        'Quantity': 'quantity',
        'UnitPrice': 'unit_price'
    }

    # Apply renaming only for columns that exist in the DataFrame
    df_transformed = df_transformed.rename(columns={k: v for k, v in column_rename_map.items() if k in df_transformed.columns})

    # Ensure all columns are lowercase after specific renames, as a fallback for any other columns
    df_transformed.columns = [col.lower() for col in df_transformed.columns]

    print(f"DEBUG: Columns after initial snake_case conversion: {df_transformed.columns.tolist()}")
    report['issues'].append('Column names standardized to snake_case.')

    # 2. Correct data types and handle missing values
    # Convert 'order_date' to datetime
    # Using dayfirst=False for 'MM/DD/YYYY' format as observed in sales_df
    df_transformed['order_date'] = pd.to_datetime(df_transformed['order_date'], errors='coerce', dayfirst=False)
    if df_transformed['order_date'].isnull().any():
        invalid_dates_count = df_transformed['order_date'].isnull().sum()
        report['issues'].append(f"Identified {invalid_dates_count} invalid 'order_date' values. Coerced to NaT.")

    # 'unit_price' is entirely NaN in the source. We will keep it as is
    # and enrich it from the products data during the integration phase.
    # Therefore, we do not reject rows based on missing 'unit_price' at this stage.
    if df_transformed['unit_price'].isnull().all():
        report['issues'].append("'unit_price' column is entirely missing in raw sales data. It will be enriched from products data during integration.")
    else:
        # If unit_price wasn't entirely NaN, try converting to numeric
        df_transformed['unit_price'] = pd.to_numeric(df_transformed['unit_price'], errors='coerce')
        if df_transformed['unit_price'].isnull().any():
            invalid_prices_count = df_transformed['unit_price'].isnull().sum() - (df_transformed['unit_price'].isnull().sum() if df['unit_price'].isnull().all() else 0)
            if invalid_prices_count > 0:
                 report['issues'].append(f"Identified {invalid_prices_count} invalid 'unit_price' values. Coerced to NaN.")

    # Ensure quantity is numeric
    df_transformed['quantity'] = pd.to_numeric(df_transformed['quantity'], errors='coerce')
    if df_transformed['quantity'].isnull().any():
        invalid_quantities_count = df_transformed['quantity'].isnull().sum()
        report['issues'].append(f"Identified {invalid_quantities_count} invalid 'quantity' values. Coerced to NaN.")

    # --- Handle potential redundant 'salesamount' column before calculating new ones ---
    if 'salesamount' in df_transformed.columns:
        df_transformed = df_transformed.drop(columns=['salesamount'])
        report['issues'].append("Removed redundant 'salesamount' column to avoid conflict with 'sales_amount'.")
    print(f"DEBUG: Columns before calculating sales_amount: {df_transformed.columns.tolist()}")

    # Calculate sales_amount and total_price. They will be NaN if unit_price or quantity is NaN.
    df_transformed['sales_amount'] = df_transformed['quantity'] * df_transformed['unit_price']
    df_transformed['total_price'] = df_transformed['sales_amount'] # Assuming total_price is sales_amount for now
    print(f"DEBUG: Columns after calculating sales_amount: {df_transformed.columns.tolist()}")


    # Check for critical missing values that would still warrant rejection
    # customer_id, product_id, quantity are critical.
    critical_missing_mask = (df_transformed['customer_id'].isnull()) | \
                            (df_transformed['product_id'].isnull()) | \
                            (df_transformed['quantity'].isnull()) | \
                            (df_transformed['order_date'].isnull())

    if critical_missing_mask.any():
        num_rejected_critical = critical_missing_mask.sum()
        rejected_critical_records = df_transformed[critical_missing_mask].to_dict(orient='records')
        rejected_records.extend(rejected_critical_records)
        df_transformed = df_transformed[~critical_missing_mask]
        report['rejected_rows'] += num_rejected_critical
        report['issues'].append(f"Rejected {num_rejected_critical} rows due to missing critical values (customer_id, product_id, quantity, or order_date).")

    report['transformed_rows'] = len(df_transformed)
    report['final_missing_values'] = df_transformed.isnull().sum().sum() # Recalculate missing values after transformations

    return df_transformed, report, rejected_records

In [13]:
# Re-run sales data transformation with the corrected function
sales_transformed_df, sales_transform_report, sales_rejected_records = transform_sales_data(sales_extracted_df)

# Update ETL logs for the re-run sales transformation
import datetime

# Initialize etl_logs if it doesn't exist
if 'etl_logs' not in globals():
    etl_logs = []

# Find the index of the previous sales ETL log and remove it if it exists
# (Assuming `etl_logs` is a list of dictionaries and `pipeline_name` exists)
initial_etl_logs_len = len(etl_logs)
etl_logs = [log for log in etl_logs if log.get('pipeline_name') != 'sales_etl']
if len(etl_logs) < initial_etl_logs_len:
    print("Removed previous 'sales_etl' entry from logs to add updated entry.")

etl_logs.append({
    'pipeline_name': 'sales_etl',
    'start_time': datetime.datetime.now(), # Placeholder, ideally from actual start
    'end_time': datetime.datetime.now(),   # Placeholder, ideally from actual end
    'rows_extracted': sales_transform_report['extracted_rows'],
    'rows_transformed': sales_transform_report['transformed_rows'],
    'rows_rejected': sales_transform_report['rejected_rows'],
    'rows_loaded': 0, # Not yet loaded
    'status': 'SUCCESS' if sales_transform_report['rejected_rows'] == 0 else 'WARNING',
    'error_message': None # Populate if actual errors occurred
})

# Display the head of the newly transformed DataFrame and the report
print("\n--- Transformed Sales Data Head ---")
display(sales_transformed_df.head())

print("\n--- Sales Transformation Report ---")
display(sales_transform_report)

print("\n--- Sales Rejected Records (if any) ---")
display(sales_rejected_records)

DEBUG: Columns after initial snake_case conversion: ['order_id', 'order_date', 'customer_id', 'product_id', 'quantity', 'unit_price', 'salesamount']
DEBUG: Columns before calculating sales_amount: ['order_id', 'order_date', 'customer_id', 'product_id', 'quantity', 'unit_price']
DEBUG: Columns after calculating sales_amount: ['order_id', 'order_date', 'customer_id', 'product_id', 'quantity', 'unit_price', 'sales_amount', 'total_price']

--- Transformed Sales Data Head ---


,order_id,order_date,customer_id,product_id,quantity,unit_price,sales_amount,total_price
0,1001,2024-12-16,711,56,4,NaN,NaN,NaN
1,1002,2023-02-18,74,987,2,NaN,NaN,NaN
2,1003,2024-02-23,283,286,9,NaN,NaN,NaN
3,1004,2024-03-27,759,41,2,NaN,NaN,NaN
4,1005,2024-08-07,208,884,10,NaN,NaN,NaN



--- Sales Transformation Report ---


{'issues': ['Column names standardized to snake_case.',
  "'unit_price' column is entirely missing in raw sales data. It will be enriched from products data during integration.",
  "Removed redundant 'salesamount' column to avoid conflict with 'sales_amount'."],
 'extracted_rows': 1000,
 'transformed_rows': 1000,
 'rejected_rows': 0,
 'data_type_errors': [],
 'final_missing_values': np.int64(3000)}


--- Sales Rejected Records (if any) ---


[]

## PHASE 2 (Revisited) — DATA QUALITY / PROFILING for `sales_transformed_df`

To explicitly address the data quality of the `sales_transformed_df` and provide a comprehensive report as per **Phase 2** of the project requirements, I will now perform a detailed analysis.

This will include:
-   Checking for missing values across all columns.
-   Identifying duplicate `order_id` values.
-   Analyzing cardinality of key identifiers.
-   Confirming data types.

The report will explicitly state that `unit_price`, `sales_amount`, and `total_price` are expected to be `NaN` at this stage, as they will be enriched during the data integration phase from the `products_transformed_df`.

In [14]:
def perform_sales_data_quality_checks(df: pd.DataFrame) -> dict:
    """
    Performs detailed data quality checks on the sales_transformed_df.
    """
    quality_report = {
        'dataset': 'sales_transformed_df',
        'total_rows': len(df),
        'issues': []
    }

    # 1. Missing Values Check
    missing_values = df.isnull().sum()
    missing_values = missing_values[missing_values > 0]
    if not missing_values.empty:
        quality_report['missing_values_summary'] = missing_values.to_dict()
        for col, count in missing_values.items():
            # Special handling for unit_price, sales_amount, total_price which are expected to be NaN
            if col in ['unit_price', 'sales_amount', 'total_price']:
                quality_report['issues'].append(
                    f"WARNING: Column '{col}' has {count} missing values. This is EXPECTED at this stage and will be enriched from products data later."
                )
            else:
                quality_report['issues'].append(
                    f"ERROR: Column '{col}' has {count} unexpected missing values."
                )
    else:
        quality_report['issues'].append("No missing values found (excluding expected NaNs).")

    # 2. Duplicate Check for OrderID
    duplicate_order_ids = df[df.duplicated(subset=['order_id'], keep=False)]
    if not duplicate_order_ids.empty:
        quality_report['duplicate_order_ids_count'] = len(duplicate_order_ids.drop_duplicates(subset=['order_id']))
        quality_report['issues'].append(
            f"ERROR: Found {quality_report['duplicate_order_ids_count']} duplicate `order_id` values. Investigate business rules for duplicates."
        )
    else:
        quality_report['issues'].append("No duplicate `order_id` values found.")

    # 3. Cardinality Check for Key Identifiers
    quality_report['cardinality'] = {
        'order_id': df['order_id'].nunique(),
        'customer_id': df['customer_id'].nunique(),
        'product_id': df['product_id'].nunique()
    }
    quality_report['issues'].append(f"Cardinality: Order IDs: {df['order_id'].nunique()}, Customer IDs: {df['customer_id'].nunique()}, Product IDs: {df['product_id'].nunique()}. ")

    # 4. Data Type Check
    dtype_report = df.dtypes.apply(lambda x: str(x)).to_dict()
    quality_report['data_types'] = dtype_report
    quality_report['issues'].append("Data types checked.")

    # Summarize overall status
    if any("ERROR" in issue for issue in quality_report['issues']):
        quality_report['status'] = 'FAIL'
    elif any("WARNING" in issue for issue in quality_report['issues']):
        quality_report['status'] = 'WARNING'
    else:
        quality_report['status'] = 'PASS'

    return quality_report

# Perform data quality checks for sales_transformed_df
sales_data_quality_report = perform_sales_data_quality_checks(sales_transformed_df)

# Display the detailed sales data quality report
print("\n--- Detailed Sales Data Quality Report ---")
for key, value in sales_data_quality_report.items():
    if isinstance(value, dict):
        print(f"{key.replace('_', ' ').title()}:")
        for sub_key, sub_value in value.items():
            print(f"  - {sub_key.replace('_', ' ').title()}: {sub_value}")
    elif isinstance(value, list):
        print(f"{key.replace('_', ' ').title()}:")
        for item in value:
            print(f"  - {item}")
    else:
        print(f"{key.replace('_', ' ').title()}: {value}")


--- Detailed Sales Data Quality Report ---
Dataset: sales_transformed_df
Total Rows: 1000
Issues:
  - WARNING: Column 'unit_price' has 1000 missing values. This is EXPECTED at this stage and will be enriched from products data later.
  - WARNING: Column 'sales_amount' has 1000 missing values. This is EXPECTED at this stage and will be enriched from products data later.
  - WARNING: Column 'total_price' has 1000 missing values. This is EXPECTED at this stage and will be enriched from products data later.
  - No duplicate `order_id` values found.
  - Cardinality: Order IDs: 1000, Customer IDs: 638, Product IDs: 614. 
  - Data types checked.
Missing Values Summary:
  - Unit Price: 1000
  - Sales Amount: 1000
  - Total Price: 1000
Cardinality:
  - Order Id: 1000
  - Customer Id: 638
  - Product Id: 614
Data Types:
  - Order Id: int64
  - Order Date: datetime64[ns]
  - Customer Id: int64
  - Product Id: int64
  - Quantity: int64
  - Unit Price: float64
  - Sales Amount: float64
  - Total P

The `sales_transformed_df` has now been thoroughly checked for data quality. The report confirms that the missing `unit_price`, `sales_amount`, and `total_price` are expected and will be handled in a later integration step. No critical issues (like unexpected missing IDs or duplicate orders) were found.

With this confirmation, we can now confidently proceed to **PHASE 4 — DATA MODELING**, where we will create the `dim_date` dimension table, followed by `dim_customer`, `dim_product`, and finally the `fact_sales` table to establish our Star Schema.

In [ ]:
# Re-adding the dim_date creation from the previous turn, as the user rejected the state of the notebook before this step.
# This ensures proper flow after data quality checks.

def create_date_dimension(sales_df: pd.DataFrame) -> pd.DataFrame:
    """
    Generates a date dimension DataFrame from the unique order dates in the sales data.
    """
    # Extract unique dates from the sales DataFrame
    dates = sales_df['order_date'].dropna().unique()
    dates_df = pd.DataFrame({'full_date': pd.to_datetime(dates)})

    # Ensure full_date is datetime type
    dates_df['full_date'] = pd.to_datetime(dates_df['full_date'])

    # Sort dates to ensure proper order
    dates_df = dates_df.sort_values('full_date').reset_index(drop=True)

    # Generate date attributes
    dates_df['date_id'] = dates_df['full_date'].dt.strftime('%Y%m%d').astype(int)
    dates_df['day_of_week'] = dates_df['full_date'].dt.day_name()
    dates_df['day_of_month'] = dates_df['full_date'].dt.day
    dates_df['day_of_year'] = dates_df['full_date'].dt.dayofyear
    dates_df['week_of_year'] = dates_df['full_date'].dt.isocalendar().week.astype(int)
    dates_df['month'] = dates_df['full_date'].dt.month_name()
    dates_df['month_num'] = dates_df['full_date'].dt.month
    dates_df['quarter'] = dates_df['full_date'].dt.quarter
    dates_df['year'] = dates_df['full_date'].dt.year
    dates_df['is_weekend'] = dates_df['full_date'].dt.weekday >= 5
    dates_df['is_holiday'] = False # Placeholder, can be enriched later
    dates_df['season'] = 'Unknown' # Placeholder, can be enriched later

    # Set date_id as index for potential future use or just keep it as a column
    dates_df = dates_df.set_index('date_id')

    # Display first few rows and info
    print("\n--- Date Dimension (dim_date) Head ---")
    display(dates_df.head())
    print("\n--- Date Dimension (dim_date) Info ---")
    dates_df.info()

    return dates_df

data_modeling_reports = []

# Generate dim_date
dim_date = create_date_dimension(sales_transformed_df)

# Record data modeling report for dim_date
data_modeling_reports.append({
    'dimension_table': 'dim_date',
    'rows_generated': len(dim_date),
    'columns_generated': len(dim_date.columns),
    'primary_key': 'date_id',
    'status': 'SUCCESS',
    'issues': []
})

# Display the report
print("\n--- Data Modeling Report for dim_date ---")
display(data_modeling_reports[-1])

In [ ]:
# ============================================================
# PHASE 6 — DATA INTEGRATION
# Enrich sales data with product information
# ============================================================

import pandas as pd
import numpy as np

# Make copies so original transformed data is not modified
sales_integrated_df = sales_transformed_df.copy()
products_for_join = products_transformed_df.copy()

# ------------------------------------------------------------
# Standardize product column names
# ------------------------------------------------------------

products_for_join.columns = [
    str(col).strip().lower().replace(" ", "_")
    for col in products_for_join.columns
]

print("Sales columns:")
print(sales_integrated_df.columns.tolist())

print("\nProduct columns:")
print(products_for_join.columns.tolist())


# ------------------------------------------------------------
# Check required columns
# ------------------------------------------------------------

required_sales_columns = ["product_id"]
required_product_columns = ["product_id"]

for col in required_sales_columns:
    if col not in sales_integrated_df.columns:
        raise KeyError(f"Missing column in sales data: {col}")

for col in required_product_columns:
    if col not in products_for_join.columns:
        raise KeyError(f"Missing column in product data: {col}")


# ------------------------------------------------------------
# Find price column automatically
# ------------------------------------------------------------

possible_price_columns = [
    "price_usd",
    "unit_price",
    "price",
    "product_price"
]

price_column = None

for col in possible_price_columns:
    if col in products_for_join.columns:
        price_column = col
        break

if price_column is None:
    raise KeyError(
        "No product price column was found in products data."
    )

print(f"\nUsing product price column: {price_column}")


# ------------------------------------------------------------
# Check duplicate Product IDs
# ------------------------------------------------------------

duplicate_products = products_for_join[
    products_for_join["product_id"].duplicated(keep=False)
]

print("\nDuplicate Product IDs:", len(duplicate_products))

if len(duplicate_products) > 0:
    print("WARNING: Duplicate product IDs found.")
    display(duplicate_products.head(10))


# ------------------------------------------------------------
# Keep only required product columns
# ------------------------------------------------------------

product_price_lookup = products_for_join[
    ["product_id", price_column]
].copy()

product_price_lookup = product_price_lookup.drop_duplicates(
    subset=["product_id"]
)


# ------------------------------------------------------------
# Rename price column
# ------------------------------------------------------------

product_price_lookup = product_price_lookup.rename(
    columns={price_column: "product_unit_price"}
)


# ------------------------------------------------------------
# Check unmatched Product IDs BEFORE joining
# ------------------------------------------------------------

sales_product_ids = set(
    sales_integrated_df["product_id"].dropna().unique()
)

product_ids = set(
    product_price_lookup["product_id"].dropna().unique()
)

unmatched_product_ids = sales_product_ids - product_ids

print(
    "\nUnmatched Product IDs:",
    len(unmatched_product_ids)
)

if len(unmatched_product_ids) > 0:
    print("Sample unmatched IDs:")
    print(list(unmatched_product_ids)[:20])


# ------------------------------------------------------------
# LEFT JOIN Sales with Products
# ------------------------------------------------------------

sales_integrated_df = sales_integrated_df.merge(
    product_price_lookup,
    on="product_id",
    how="left"
)


# ------------------------------------------------------------
# Fill missing unit_price from Product table
# ------------------------------------------------------------

sales_integrated_df["unit_price"] = (
    sales_integrated_df["unit_price"]
    .fillna(sales_integrated_df["product_unit_price"])
)


# ------------------------------------------------------------
# Recalculate sales amount
# ------------------------------------------------------------

sales_integrated_df["sales_amount"] = (
    sales_integrated_df["quantity"]
    * sales_integrated_df["unit_price"]
)

sales_integrated_df["total_price"] = (
    sales_integrated_df["sales_amount"]
)


# ------------------------------------------------------------
# Remove temporary lookup column
# ------------------------------------------------------------

sales_integrated_df = sales_integrated_df.drop(
    columns=["product_unit_price"]
)


# ------------------------------------------------------------
# Integration validation
# ------------------------------------------------------------

print("\n============================================")
print("INTEGRATION RESULT")
print("============================================")

print("Rows before integration:", len(sales_transformed_df))
print("Rows after integration :", len(sales_integrated_df))

print(
    "Missing unit_price after enrichment:",
    sales_integrated_df["unit_price"].isna().sum()
)

print(
    "Missing sales_amount:",
    sales_integrated_df["sales_amount"].isna().sum()
)

print(
    "Total Sales:",
    sales_integrated_df["sales_amount"].sum()
)

display(sales_integrated_df.head())

In [ ]:
# ============================================================
# PHASE 6.2 — CREATE INTEGRATED ANALYTICS DATASET
# ============================================================

analytics_df = sales_integrated_df.copy()

customers_for_join = customers_transformed_df.copy()
products_for_join = products_transformed_df.copy()

# Standardize column names
customers_for_join.columns = [
    str(col).strip().lower().replace(" ", "_")
    for col in customers_for_join.columns
]

products_for_join.columns = [
    str(col).strip().lower().replace(" ", "_")
    for col in products_for_join.columns
]

# ------------------------------------------------------------
# Customer columns
# ------------------------------------------------------------

print("Customer columns:")
print(customers_for_join.columns.tolist())

print("\nProduct columns:")
print(products_for_join.columns.tolist())


# ------------------------------------------------------------
# Remove duplicate dimension IDs
# ------------------------------------------------------------

customers_for_join = customers_for_join.drop_duplicates(
    subset=["customer_id"]
)

products_for_join = products_for_join.drop_duplicates(
    subset=["product_id"]
)


# ------------------------------------------------------------
# Join Customer information
# ------------------------------------------------------------

analytics_df = analytics_df.merge(
    customers_for_join,
    on="customer_id",
    how="left",
    suffixes=("", "_customer")
)


# ------------------------------------------------------------
# Join Product information
# ------------------------------------------------------------

analytics_df = analytics_df.merge(
    products_for_join,
    on="product_id",
    how="left",
    suffixes=("", "_product")
)


print("\n============================================")
print("ANALYTICS DATASET")
print("============================================")

print("Rows:", len(analytics_df))
print("Columns:", len(analytics_df.columns))

display(analytics_df.head())

In [ ]:
# ============================================================
# PHASE 7 — DATE DIMENSION
# ============================================================

date_min = analytics_df["order_date"].min()
date_max = analytics_df["order_date"].max()

print("Date range:")
print("Start:", date_min)
print("End  :", date_max)


# Create complete date range
date_range = pd.date_range(
    start=date_min,
    end=date_max,
    freq="D"
)

dim_date = pd.DataFrame({
    "full_date": date_range
})

# Surrogate key
dim_date["date_key"] = (
    dim_date["full_date"].dt.strftime("%Y%m%d").astype(int)
)

dim_date["day"] = dim_date["full_date"].dt.day
dim_date["month"] = dim_date["full_date"].dt.month
dim_date["month_name"] = dim_date["full_date"].dt.month_name()
dim_date["quarter"] = dim_date["full_date"].dt.quarter
dim_date["year"] = dim_date["full_date"].dt.year
dim_date["day_of_week"] = dim_date["full_date"].dt.dayofweek + 1
dim_date["day_name"] = dim_date["full_date"].dt.day_name()
dim_date["week_number"] = dim_date["full_date"].dt.isocalendar().week.astype(int)

# Reorder columns
dim_date = dim_date[
    [
        "date_key",
        "full_date",
        "day",
        "month",
        "month_name",
        "quarter",
        "year",
        "day_of_week",
        "day_name",
        "week_number"
    ]
]

print("\nDate dimension created:")
print(dim_date.shape)

display(dim_date.head())

In [ ]:
# ============================================================
# PHASE 8 — FACT SALES
# ============================================================

import pandas as pd
import numpy as np

# ------------------------------------------------------------
# Create a copy of integrated sales data
# ------------------------------------------------------------

fact_sales = analytics_df.copy()


# ------------------------------------------------------------
# Make sure order_date is datetime
# ------------------------------------------------------------

fact_sales["order_date"] = pd.to_datetime(
    fact_sales["order_date"],
    errors="coerce"
)


# ------------------------------------------------------------
# Create DATE KEY
# Format: YYYYMMDD
# ------------------------------------------------------------

fact_sales["date_key"] = (
    fact_sales["order_date"]
    .dt.strftime("%Y%m%d")
)

# Convert to numeric
fact_sales["date_key"] = pd.to_numeric(
    fact_sales["date_key"],
    errors="coerce"
).astype("Int64")


# ------------------------------------------------------------
# CREATE CUSTOMER DIMENSION
# ------------------------------------------------------------

dim_customer = (
    fact_sales[["customer_id"]]
    .drop_duplicates()
    .sort_values("customer_id")
    .reset_index(drop=True)
)

# Create surrogate customer key
dim_customer.insert(
    0,
    "customer_key",
    range(1, len(dim_customer) + 1)
)


# ------------------------------------------------------------
# CREATE PRODUCT DIMENSION
# ------------------------------------------------------------

dim_product = (
    fact_sales[["product_id"]]
    .drop_duplicates()
    .sort_values("product_id")
    .reset_index(drop=True)
)

# Create surrogate product key
dim_product.insert(
    0,
    "product_key",
    range(1, len(dim_product) + 1)
)


# ------------------------------------------------------------
# CREATE LOOKUP DICTIONARIES
# ------------------------------------------------------------

customer_key_map = dict(
    zip(
        dim_customer["customer_id"],
        dim_customer["customer_key"]
    )
)

product_key_map = dict(
    zip(
        dim_product["product_id"],
        dim_product["product_key"]
    )
)


# ------------------------------------------------------------
# MAP CUSTOMER SURROGATE KEY
# ------------------------------------------------------------

fact_sales["customer_key"] = (
    fact_sales["customer_id"]
    .map(customer_key_map)
)


# ------------------------------------------------------------
# MAP PRODUCT SURROGATE KEY
# ------------------------------------------------------------

fact_sales["product_key"] = (
    fact_sales["product_id"]
    .map(product_key_map)
)


# ------------------------------------------------------------
# SELECT FACT TABLE COLUMNS
# ------------------------------------------------------------

fact_columns = [
    "order_id",
    "customer_key",
    "product_key",
    "date_key",
    "quantity",
    "unit_price",
    "sales_amount",
    "total_price"
]

# Keep only columns that actually exist
fact_columns = [
    col for col in fact_columns
    if col in fact_sales.columns
]

fact_sales = fact_sales[fact_columns].copy()


# ------------------------------------------------------------
# VALIDATION
# ------------------------------------------------------------

print("=" * 60)
print("FACT SALES CREATED")
print("=" * 60)

print("Fact Sales Rows    :", len(fact_sales))
print("Fact Sales Columns :", len(fact_sales.columns))

print("\nDimension Sizes")
print("-" * 40)

print("Customers :", len(dim_customer))
print("Products  :", len(dim_product))


# ------------------------------------------------------------
# CHECK NULL FOREIGN KEYS
# ------------------------------------------------------------

print("\nForeign Key Validation")
print("-" * 40)

print(
    "Missing customer_key :",
    fact_sales["customer_key"].isna().sum()
)

print(
    "Missing product_key  :",
    fact_sales["product_key"].isna().sum()
)

print(
    "Missing date_key     :",
    fact_sales["date_key"].isna().sum()
)


# ------------------------------------------------------------
# CHECK SALES VALUES
# ------------------------------------------------------------

print("\nSales Validation")
print("-" * 40)

print(
    "Total Quantity :",
    fact_sales["quantity"].sum()
)

print(
    "Total Sales    :",
    fact_sales["sales_amount"].sum()
)


# ------------------------------------------------------------
# DISPLAY RESULTS
# ------------------------------------------------------------

print("\n--- FACT SALES ---")
display(fact_sales.head())

print("\n--- DIM CUSTOMER ---")
display(dim_customer.head())

print("\n--- DIM PRODUCT ---")
display(dim_product.head())

In [ ]:
# ============================================================
# PHASE 9 — LOAD
# ============================================================

import os

OUTPUT_DIR = "/content/etl_output"

os.makedirs(
    OUTPUT_DIR,
    exist_ok=True
)


# ------------------------------------------------------------
# Save cleaned datasets
# ------------------------------------------------------------

clean_sales_path = f"{OUTPUT_DIR}/cleaned_sales.csv"
clean_customers_path = f"{OUTPUT_DIR}/cleaned_customers.csv"
clean_products_path = f"{OUTPUT_DIR}/cleaned_products.csv"

sales_integrated_df.to_csv(
    clean_sales_path,
    index=False
)

customers_for_join.to_csv(
    clean_customers_path,
    index=False
)

products_for_join.to_csv(
    clean_products_path,
    index=False
)


# ------------------------------------------------------------
# Save dimension tables
# ------------------------------------------------------------

dim_customer.to_csv(
    f"{OUTPUT_DIR}/dim_customer.csv",
    index=False
)

dim_product.to_csv(
    f"{OUTPUT_DIR}/dim_product.csv",
    index=False
)

dim_date.to_csv(
    f"{OUTPUT_DIR}/dim_date.csv",
    index=False
)


# ------------------------------------------------------------
# Save fact table
# ------------------------------------------------------------

fact_sales.to_csv(
    f"{OUTPUT_DIR}/fact_sales.csv",
    index=False
)


# ------------------------------------------------------------
# Save final analytics dataset
# ------------------------------------------------------------

analytics_df.to_csv(
    f"{OUTPUT_DIR}/final_sales_analytics.csv",
    index=False
)


print("============================================")
print("LOAD COMPLETED")
print("============================================")

for file in os.listdir(OUTPUT_DIR):
    print(file)

In [ ]:
# ============================================================
# PHASE 10 — FINAL ETL VALIDATION
# ============================================================

validation_results = []


def add_validation(check, expected, actual):
    if expected == actual:
        status = "PASS"
    else:
        status = "WARNING"

    validation_results.append({
        "Check": check,
        "Expected": expected,
        "Actual": actual,
        "Status": status
    })


# ------------------------------------------------------------
# Row count validation
# ------------------------------------------------------------

add_validation(
    "Sales row count",
    len(sales_transformed_df),
    len(fact_sales)
)


# ------------------------------------------------------------
# Primary key validation
# ------------------------------------------------------------

add_validation(
    "Unique Order IDs",
    fact_sales["order_id"].nunique(),
    len(fact_sales)
)


# ------------------------------------------------------------
# Missing values
# ------------------------------------------------------------

add_validation(
    "Missing Unit Price",
    0,
    fact_sales["unit_price"].isna().sum()
)

add_validation(
    "Missing Sales Amount",
    0,
    fact_sales["sales_amount"].isna().sum()
)


# ------------------------------------------------------------
# Dimension validation
# ------------------------------------------------------------

add_validation(
    "Customer dimension mapping",
    0,
    fact_sales["customer_key"].isna().sum()
)

add_validation(
    "Product dimension mapping",
    0,
    fact_sales["product_key"].isna().sum()
)


# ------------------------------------------------------------
# Display validation report
# ------------------------------------------------------------

validation_df = pd.DataFrame(validation_results)

display(validation_df)


# Save validation report
validation_df.to_csv(
    f"{OUTPUT_DIR}/etl_quality_report.csv",
    index=False
)


# ------------------------------------------------------------
# Final summary
# ------------------------------------------------------------

print("\n============================================")
print("FINAL ETL SUMMARY")
print("============================================")

print("Raw Sales Rows      :", len(sales_extracted_df))
print("Transformed Rows    :", len(sales_transformed_df))
print("Fact Sales Rows     :", len(fact_sales))
print("Customers           :", fact_sales["customer_key"].nunique())
print("Products            :", fact_sales["product_key"].nunique())
print("Total Quantity      :", fact_sales["quantity"].sum())
print("Total Sales         :", fact_sales["sales_amount"].sum())
print("Missing Values      :", fact_sales.isna().sum().sum())
print("Duplicate Orders    :", fact_sales["order_id"].duplicated().sum())

if (validation_df["Status"] == "FAIL").any():
    print("\nETL PIPELINE STATUS: FAIL")
elif (validation_df["Status"] == "WARNING").any():
    print("\nETL PIPELINE STATUS: WARNING")
else:
    print("\nETL PIPELINE STATUS: PASS")

print("\nETL PIPELINE COMPLETED")

In [ ]:
# ============================================================
# PHASE 6.1 — PRODUCT INTEGRATION (FIXED & ROBUST)
# ============================================================

import pandas as pd
import numpy as np

print("=" * 60)
print("PHASE 6.1 — PRODUCT INTEGRATION")
print("=" * 60)

# ------------------------------------------------------------
# 1. Check required DataFrames
# ------------------------------------------------------------

if "sales_transformed_df" not in globals():
    raise NameError("sales_transformed_df not found. Run the Sales Transformation phase first.")

if "products_transformed_df" not in globals():
    raise NameError(
        "products_transformed_df not found. "
        "Run the Products Transformation phase before Phase 6.1."
    )

sales_integrated = sales_transformed_df.copy()
products_lookup = products_transformed_df.copy()

print(f"Sales rows: {len(sales_integrated):,}")
print(f"Product rows: {len(products_lookup):,}")

print("\nSales columns:")
print(sales_integrated.columns.tolist())

print("\nProduct columns:")
print(products_lookup.columns.tolist())


# ------------------------------------------------------------
# 2. Standardize column names
# ------------------------------------------------------------

sales_integrated.columns = (
    sales_integrated.columns
    .astype(str)
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)

products_lookup.columns = (
    products_lookup.columns
    .astype(str)
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)


# ------------------------------------------------------------
# 3. Detect Product ID column
# ------------------------------------------------------------

product_id_candidates = [
    "product_id",
    "productid",
    "product"
]

product_id_col = next(
    (c for c in product_id_candidates if c in products_lookup.columns),
    None
)

if "product_id" not in sales_integrated.columns:
    # Try to find equivalent sales product column
    sales_product_col = next(
        (c for c in product_id_candidates if c in sales_integrated.columns),
        None
    )

    if sales_product_col:
        sales_integrated.rename(
            columns={sales_product_col: "product_id"},
            inplace=True
        )
    else:
        raise ValueError(
            "Sales dataset does not contain a Product ID column."
        )

if product_id_col is None:
    raise ValueError(
        "Products dataset does not contain a Product ID column."
    )

if product_id_col != "product_id":
    products_lookup.rename(
        columns={product_id_col: "product_id"},
        inplace=True
    )


# ------------------------------------------------------------
# 4. Detect price column automatically
# ------------------------------------------------------------

price_candidates = [
    "price_usd",
    "unit_price",
    "price",
    "product_price",
    "selling_price",
    "cost",
    "amount"
]

price_col = next(
    (c for c in price_candidates if c in products_lookup.columns),
    None
)

if price_col is None:
    raise ValueError(
        "Could not find a product price column.\n"
        f"Available product columns: {products_lookup.columns.tolist()}\n"
        f"Expected one of: {price_candidates}"
    )

print(f"\nProduct ID column: {product_id_col if product_id_col != 'product_id' else 'product_id'}")
print(f"Price column detected: {price_col}")


# ------------------------------------------------------------
# 5. Clean keys
# ------------------------------------------------------------

sales_integrated["product_id"] = (
    sales_integrated["product_id"]
    .astype(str)
    .str.strip()
)

products_lookup["product_id"] = (
    products_lookup["product_id"]
    .astype(str)
    .str.strip()
)


# ------------------------------------------------------------
# 6. Convert price to numeric
# ------------------------------------------------------------

products_lookup[price_col] = pd.to_numeric(
    products_lookup[price_col],
    errors="coerce"
)

sales_integrated["quantity"] = pd.to_numeric(
    sales_integrated["quantity"],
    errors="coerce"
)


# ------------------------------------------------------------
# 7. Remove invalid product IDs from lookup
# ------------------------------------------------------------

products_lookup = products_lookup[
    products_lookup["product_id"].notna() &
    (products_lookup["product_id"] != "") &
    (products_lookup["product_id"].str.lower() != "nan")
].copy()


# ------------------------------------------------------------
# 8. Check duplicate product IDs
# ------------------------------------------------------------

duplicate_products = products_lookup[
    products_lookup["product_id"].duplicated(keep=False)
]

print(f"\nDuplicate product IDs: {len(duplicate_products):,}")

if not duplicate_products.empty:
    print("WARNING: Duplicate Product IDs found.")
    print("Keeping first record for each Product ID.")

    products_lookup = products_lookup.drop_duplicates(
        subset=["product_id"],
        keep="first"
    )


# ------------------------------------------------------------
# 9. Check product matching
# ------------------------------------------------------------

matched_mask = sales_integrated["product_id"].isin(
    products_lookup["product_id"]
)

matched_count = int(matched_mask.sum())
unmatched_count = int((~matched_mask).sum())
total_rows = len(sales_integrated)

match_percentage = (
    matched_count / total_rows * 100
    if total_rows > 0 else 0
)

print("\n--- Product Matching Results ---")
print(f"Total sales rows       : {total_rows:,}")
print(f"Matched product rows   : {matched_count:,}")
print(f"Unmatched product rows : {unmatched_count:,}")
print(f"Match percentage       : {match_percentage:.2f}%")


# ------------------------------------------------------------
# 10. Create product price lookup
# ------------------------------------------------------------

product_price_lookup = products_lookup[
    ["product_id", price_col]
].copy()

product_price_lookup.rename(
    columns={price_col: "product_price"},
    inplace=True
)


# ------------------------------------------------------------
# 11. Join product price
# ------------------------------------------------------------

sales_integrated = sales_integrated.merge(
    product_price_lookup,
    on="product_id",
    how="left",
    validate="many_to_one"
)


# ------------------------------------------------------------
# 12. Populate unit_price
# ------------------------------------------------------------

# Use existing unit_price if available,
# otherwise use product price.

if "unit_price" not in sales_integrated.columns:
    sales_integrated["unit_price"] = np.nan

sales_integrated["unit_price"] = pd.to_numeric(
    sales_integrated["unit_price"],
    errors="coerce"
)

sales_integrated["product_price"] = pd.to_numeric(
    sales_integrated["product_price"],
    errors="coerce"
)

sales_integrated["unit_price"] = (
    sales_integrated["unit_price"]
    .fillna(sales_integrated["product_price"])
)


# ------------------------------------------------------------
# 13. Calculate Sales Amount
# ------------------------------------------------------------

sales_integrated["sales_amount"] = (
    sales_integrated["quantity"] *
    sales_integrated["unit_price"]
)


# ------------------------------------------------------------
# 14. Total Price
# ------------------------------------------------------------

sales_integrated["total_price"] = (
    sales_integrated["sales_amount"]
)


# ------------------------------------------------------------
# 15. Remove temporary column
# ------------------------------------------------------------

sales_integrated.drop(
    columns=["product_price"],
    inplace=True,
    errors="ignore"
)


# ------------------------------------------------------------
# 16. Final product integration checks
# ------------------------------------------------------------

missing_unit_price = int(
    sales_integrated["unit_price"].isna().sum()
)

missing_sales_amount = int(
    sales_integrated["sales_amount"].isna().sum()
)

print("\n" + "=" * 60)
print("PRODUCT INTEGRATION COMPLETED")
print("=" * 60)

print(f"Rows after integration : {len(sales_integrated):,}")
print(f"Missing Unit Price     : {missing_unit_price:,}")
print(f"Missing Sales Amount   : {missing_sales_amount:,}")

print("\nIntegrated Sales Data:")
display(sales_integrated.head())

print("\nData Types:")
display(sales_integrated.dtypes)

In [ ]:
# ============================================================
# PHASE 6.2 — CUSTOMER INTEGRATION
# ============================================================

print("=" * 60)
print("PHASE 6.2 — CUSTOMER INTEGRATION")
print("=" * 60)

if "customers_transformed_df" not in globals():
    raise NameError(
        "customers_transformed_df not found. "
        "Run the Customers Transformation phase first."
    )

sales_analytics = sales_integrated.copy()
customers_lookup = customers_transformed_df.copy()

# Standardize column names
sales_analytics.columns = (
    sales_analytics.columns.astype(str)
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)

customers_lookup.columns = (
    customers_lookup.columns.astype(str)
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)

# Detect customer ID
customer_candidates = [
    "customer_id",
    "customerid",
    "customer"
]

customer_id_col = next(
    (c for c in customer_candidates if c in customers_lookup.columns),
    None
)

if customer_id_col is None:
    raise ValueError(
        f"Customer ID not found. Available columns: "
        f"{customers_lookup.columns.tolist()}"
    )

if "customer_id" not in sales_analytics.columns:
    sales_customer_col = next(
        (c for c in customer_candidates if c in sales_analytics.columns),
        None
    )

    if sales_customer_col:
        sales_analytics.rename(
            columns={sales_customer_col: "customer_id"},
            inplace=True
        )
    else:
        raise ValueError("customer_id not found in sales data.")

if customer_id_col != "customer_id":
    customers_lookup.rename(
        columns={customer_id_col: "customer_id"},
        inplace=True
    )

# Clean keys
sales_analytics["customer_id"] = (
    sales_analytics["customer_id"]
    .astype(str)
    .str.strip()
)

customers_lookup["customer_id"] = (
    customers_lookup["customer_id"]
    .astype(str)
    .str.strip()
)

# Remove duplicate customers
duplicate_customers = customers_lookup[
    customers_lookup["customer_id"].duplicated(keep=False)
]

print(f"Duplicate customer IDs: {len(duplicate_customers):,}")

customers_lookup = customers_lookup.drop_duplicates(
    subset=["customer_id"],
    keep="first"
)

# Matching
customer_match = sales_analytics["customer_id"].isin(
    customers_lookup["customer_id"]
)

print(f"Total sales rows       : {len(sales_analytics):,}")
print(f"Matched customer rows  : {customer_match.sum():,}")
print(f"Unmatched customer rows: {(~customer_match).sum():,}")

# Join
customer_columns = [
    c for c in customers_lookup.columns
    if c != "customer_id"
]

sales_analytics = sales_analytics.merge(
    customers_lookup[
        ["customer_id"] + customer_columns
    ],
    on="customer_id",
    how="left",
    validate="many_to_one",
    suffixes=("", "_customer")
)

print("\nCustomer integration completed.")
display(sales_analytics.head())

In [ ]:
# ============================================================
# PHASE 6.3 — DATE INTEGRATION (FIXED)
# ============================================================

print("=" * 60)
print("PHASE 6.3 — DATE INTEGRATION")
print("=" * 60)

# ------------------------------------------------------------
# 1. Check required DataFrames
# ------------------------------------------------------------

if "sales_analytics" not in globals():
    raise NameError(
        "sales_analytics not found. Run Phase 6.2 first."
    )

if "dim_date" not in globals():
    raise NameError(
        "dim_date not found. Run the Date Dimension phase first."
    )


# ------------------------------------------------------------
# 2. Work on copy
# ------------------------------------------------------------

sales_analytics = sales_analytics.copy()


# ------------------------------------------------------------
# 3. Clean order_date
# ------------------------------------------------------------

if "order_date" not in sales_analytics.columns:
    raise ValueError(
        "order_date column is missing from sales_analytics."
    )

sales_analytics["order_date"] = pd.to_datetime(
    sales_analytics["order_date"],
    errors="coerce"
).dt.normalize()


# ------------------------------------------------------------
# 4. Prepare Date Dimension
# ------------------------------------------------------------

# dim_date may have date_id as index
date_lookup = dim_date.copy()

# If date_id is index, reset it
if "date_id" not in date_lookup.columns:
    date_lookup = date_lookup.reset_index()

# If reset_index created a different index column name
if "date_id" not in date_lookup.columns:
    # Try to identify the first column as date key
    possible_key = [
        c for c in date_lookup.columns
        if str(c).lower() in ["date_id", "date_key"]
    ]

    if possible_key:
        date_lookup.rename(
            columns={possible_key[0]: "date_id"},
            inplace=True
        )
    else:
        raise ValueError(
            "Could not find date_id in dim_date."
        )


# ------------------------------------------------------------
# 5. Find date column
# ------------------------------------------------------------

date_candidates = [
    "full_date",
    "date",
    "calendar_date",
    "order_date"
]

date_column = next(
    (
        c for c in date_candidates
        if c in date_lookup.columns
    ),
    None
)

if date_column is None:
    raise ValueError(
        "Could not find date column in dim_date.\n"
        f"Available columns: {date_lookup.columns.tolist()}"
    )


# Rename date column to full_date
if date_column != "full_date":
    date_lookup.rename(
        columns={date_column: "full_date"},
        inplace=True
    )


# ------------------------------------------------------------
# 6. Clean Date Dimension
# ------------------------------------------------------------

date_lookup["full_date"] = pd.to_datetime(
    date_lookup["full_date"],
    errors="coerce"
).dt.normalize()

date_lookup["date_id"] = pd.to_numeric(
    date_lookup["date_id"],
    errors="coerce"
)


# ------------------------------------------------------------
# 7. Remove invalid / duplicate dates
# ------------------------------------------------------------

date_lookup = date_lookup[
    date_lookup["full_date"].notna()
].copy()

date_lookup = date_lookup.drop_duplicates(
    subset=["full_date"],
    keep="first"
)


# ------------------------------------------------------------
# 8. Check date matching
# ------------------------------------------------------------

date_match = sales_analytics["order_date"].isin(
    date_lookup["full_date"]
)

total_rows = len(sales_analytics)
matched_rows = int(date_match.sum())
unmatched_rows = int((~date_match).sum())

match_percentage = (
    matched_rows / total_rows * 100
    if total_rows > 0 else 0
)

print("\n--- DATE MATCHING ---")
print(f"Sales rows        : {total_rows:,}")
print(f"Matched dates     : {matched_rows:,}")
print(f"Unmatched dates   : {unmatched_rows:,}")
print(f"Match percentage  : {match_percentage:.2f}%")


# ------------------------------------------------------------
# 9. Remove old date-dimension columns if they already exist
# ------------------------------------------------------------

date_dimension_columns = [
    "date_id",
    "full_date",
    "year",
    "month",
    "month_name",
    "quarter",
    "day",
    "day_of_week",
    "week",
    "is_weekend"
]

# Only remove columns that already exist
# This prevents _x / _y duplicate problems
for col in date_dimension_columns:
    if col in sales_analytics.columns:
        sales_analytics.drop(
            columns=[col],
            inplace=True
        )


# ------------------------------------------------------------
# 10. Join Date Dimension
# ------------------------------------------------------------

date_columns_to_add = [
    c for c in date_lookup.columns
    if c != "full_date"
]

date_join = date_lookup[
    ["full_date"] + date_columns_to_add
].copy()

sales_analytics = sales_analytics.merge(
    date_join,
    left_on="order_date",
    right_on="full_date",
    how="left",
    validate="many_to_one"
)


# ------------------------------------------------------------
# 11. Final Date Columns
# ------------------------------------------------------------

if "year" not in sales_analytics.columns:
    sales_analytics["year"] = (
        sales_analytics["order_date"].dt.year
    )

if "month" not in sales_analytics.columns:
    sales_analytics["month"] = (
        sales_analytics["order_date"].dt.month
    )

if "quarter" not in sales_analytics.columns:
    sales_analytics["quarter"] = (
        sales_analytics["order_date"].dt.quarter
    )


# ------------------------------------------------------------
# 12. Remove duplicate helper column
# ------------------------------------------------------------

if "full_date" in sales_analytics.columns:
    sales_analytics.drop(
        columns=["full_date"],
        inplace=True
    )


# ------------------------------------------------------------
# 13. Results
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("DATE INTEGRATION COMPLETED")
print("=" * 60)

print(f"Rows after date join : {len(sales_analytics):,}")
print(
    f"Missing date_id     : "
    f"{sales_analytics['date_id'].isna().sum():,}"
)

display(
    sales_analytics[
        [
            "order_id",
            "order_date",
            "date_id",
            "year",
            "month",
            "quarter"
        ]
    ].head()
)

In [ ]:
# ============================================================
# PHASE 7.1 — DIM CUSTOMER
# ============================================================

print("=" * 60)
print("PHASE 7.1 — DIM CUSTOMER")
print("=" * 60)

if "customers_lookup" not in globals():
    raise NameError(
        "customers_lookup not found. Run Phase 6.2 first."
    )

dim_customer = customers_lookup.copy()

# Clean column names
dim_customer.columns = (
    dim_customer.columns
    .astype(str)
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)

if "customer_id" not in dim_customer.columns:
    raise ValueError(
        "customer_id not found in customers_lookup."
    )

dim_customer["customer_id"] = (
    dim_customer["customer_id"]
    .astype(str)
    .str.strip()
)

# Remove invalid IDs
dim_customer = dim_customer[
    dim_customer["customer_id"].notna() &
    (dim_customer["customer_id"] != "") &
    (dim_customer["customer_id"].str.lower() != "nan")
].copy()

# Remove duplicates
dim_customer = dim_customer.drop_duplicates(
    subset=["customer_id"],
    keep="first"
).reset_index(drop=True)

# Surrogate key
dim_customer.insert(
    0,
    "customer_key",
    range(1, len(dim_customer) + 1)
)

print(f"Dimension Customer rows: {len(dim_customer):,}")

display(dim_customer.head())

In [ ]:
# ============================================================
# PHASE 7.2 — DIM PRODUCT
# ============================================================

print("=" * 60)
print("PHASE 7.2 — DIM PRODUCT")
print("=" * 60)

if "products_lookup" not in globals():
    raise NameError(
        "products_lookup not found. Run Phase 6.1 first."
    )

dim_product = products_lookup.copy()

# Clean column names
dim_product.columns = (
    dim_product.columns
    .astype(str)
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)

if "product_id" not in dim_product.columns:
    raise ValueError(
        "product_id not found in products_lookup."
    )

dim_product["product_id"] = (
    dim_product["product_id"]
    .astype(str)
    .str.strip()
)

# Remove invalid IDs
dim_product = dim_product[
    dim_product["product_id"].notna() &
    (dim_product["product_id"] != "") &
    (dim_product["product_id"].str.lower() != "nan")
].copy()

# Remove duplicates
dim_product = dim_product.drop_duplicates(
    subset=["product_id"],
    keep="first"
).reset_index(drop=True)

# Surrogate key
dim_product.insert(
    0,
    "product_key",
    range(1, len(dim_product) + 1)
)

print(f"Dimension Product rows: {len(dim_product):,}")

display(dim_product.head())

In [ ]:
# ============================================================
# PHASE 8 — FACT SALES
# ============================================================

print("=" * 60)
print("PHASE 8 — FACT SALES")
print("=" * 60)

fact_sales = sales_analytics.copy()


# ------------------------------------------------------------
# 1. Customer Key Mapping
# ------------------------------------------------------------

customer_map = dim_customer[
    ["customer_id", "customer_key"]
].copy()

customer_map["customer_id"] = (
    customer_map["customer_id"]
    .astype(str)
    .str.strip()
)

fact_sales["customer_id"] = (
    fact_sales["customer_id"]
    .astype(str)
    .str.strip()
)

fact_sales = fact_sales.merge(
    customer_map,
    on="customer_id",
    how="left",
    validate="many_to_one"
)


# ------------------------------------------------------------
# 2. Product Key Mapping
# ------------------------------------------------------------

product_map = dim_product[
    ["product_id", "product_key"]
].copy()

product_map["product_id"] = (
    product_map["product_id"]
    .astype(str)
    .str.strip()
)

fact_sales["product_id"] = (
    fact_sales["product_id"]
    .astype(str)
    .str.strip()
)

fact_sales = fact_sales.merge(
    product_map,
    on="product_id",
    how="left",
    validate="many_to_one"
)


# ------------------------------------------------------------
# 3. Date Key
# ------------------------------------------------------------

if "date_id" not in fact_sales.columns:
    raise ValueError(
        "date_id not found. Run Phase 6.3 first."
    )

fact_sales["date_key"] = pd.to_numeric(
    fact_sales["date_id"],
    errors="coerce"
)


# ------------------------------------------------------------
# 4. Numeric columns
# ------------------------------------------------------------

for col in [
    "quantity",
    "unit_price",
    "sales_amount",
    "total_price"
]:
    if col in fact_sales.columns:
        fact_sales[col] = pd.to_numeric(
            fact_sales[col],
            errors="coerce"
        )


# ------------------------------------------------------------
# 5. Select Fact Columns
# ------------------------------------------------------------

required_fact_columns = [
    "order_id",
    "customer_key",
    "product_key",
    "date_key",
    "quantity",
    "unit_price",
    "sales_amount",
    "total_price"
]

missing_fact_columns = [
    c for c in required_fact_columns
    if c not in fact_sales.columns
]

if missing_fact_columns:
    raise ValueError(
        f"Missing fact columns: {missing_fact_columns}"
    )

fact_sales = fact_sales[
    required_fact_columns
].copy()


# ------------------------------------------------------------
# 6. Validate Foreign Keys
# ------------------------------------------------------------

missing_customer_fk = int(
    fact_sales["customer_key"].isna().sum()
)

missing_product_fk = int(
    fact_sales["product_key"].isna().sum()
)

missing_date_fk = int(
    fact_sales["date_key"].isna().sum()
)

print("\n--- FACT TABLE VALIDATION ---")

print(
    f"Fact rows             : "
    f"{len(fact_sales):,}"
)

print(
    f"Missing customer FK   : "
    f"{missing_customer_fk:,}"
)

print(
    f"Missing product FK    : "
    f"{missing_product_fk:,}"
)

print(
    f"Missing date FK       : "
    f"{missing_date_fk:,}"
)

display(fact_sales.head())

In [ ]:
# ============================================================
# PHASE 9 — FINAL SALES ANALYTICS
# ============================================================

print("=" * 60)
print("PHASE 9 — FINAL SALES ANALYTICS")
print("=" * 60)

final_sales_analytics = sales_analytics.copy()

# Date
final_sales_analytics["order_date"] = pd.to_datetime(
    final_sales_analytics["order_date"],
    errors="coerce"
)

# Year
final_sales_analytics["sales_year"] = (
    final_sales_analytics["order_date"].dt.year
)

# Month number
final_sales_analytics["sales_month_num"] = (
    final_sales_analytics["order_date"].dt.month
)

# Month name
final_sales_analytics["sales_month"] = (
    final_sales_analytics["order_date"]
    .dt.strftime("%B")
)

# Quarter
final_sales_analytics["sales_quarter"] = (
    "Q" +
    final_sales_analytics["order_date"]
    .dt.quarter
    .astype("Int64")
    .astype(str)
)

# Numeric conversion
final_sales_analytics["sales_amount"] = pd.to_numeric(
    final_sales_analytics["sales_amount"],
    errors="coerce"
)

final_sales_analytics["quantity"] = pd.to_numeric(
    final_sales_analytics["quantity"],
    errors="coerce"
)

# Order total
final_sales_analytics["order_total"] = (
    final_sales_analytics
    .groupby("order_id")["sales_amount"]
    .transform("sum")
)

print(f"Final analytics rows: {len(final_sales_analytics):,}")

display(final_sales_analytics.head())

In [ ]:
# ============================================================
# PHASE 10 — BUSINESS ANALYSIS
# ============================================================

print("=" * 60)
print("PHASE 10 — BUSINESS ANALYSIS")
print("=" * 60)

total_sales = (
    final_sales_analytics["sales_amount"]
    .sum()
)

total_quantity = (
    final_sales_analytics["quantity"]
    .sum()
)

total_orders = (
    final_sales_analytics["order_id"]
    .nunique()
)

unique_customers = (
    final_sales_analytics["customer_id"]
    .nunique()
)

unique_products = (
    final_sales_analytics["product_id"]
    .nunique()
)

average_order_value = (
    total_sales / total_orders
    if total_orders > 0 else 0
)


# ------------------------------------------------------------
# KPIs
# ------------------------------------------------------------

print("\n--- KEY BUSINESS METRICS ---")

print(f"Total Sales         : ${total_sales:,.2f}")
print(f"Total Quantity      : {total_quantity:,.0f}")
print(f"Total Orders        : {total_orders:,}")
print(f"Unique Customers    : {unique_customers:,}")
print(f"Unique Products     : {unique_products:,}")
print(f"Average Order Value : ${average_order_value:,.2f}")


# ------------------------------------------------------------
# Product Analysis
# ------------------------------------------------------------

sales_by_product = (
    final_sales_analytics
    .groupby("product_id", as_index=False)
    .agg(
        total_sales=("sales_amount", "sum"),
        total_quantity=("quantity", "sum"),
        order_count=("order_id", "nunique")
    )
    .sort_values(
        "total_sales",
        ascending=False
    )
)

print("\n--- TOP 10 PRODUCTS ---")
display(sales_by_product.head(10))


# ------------------------------------------------------------
# Customer Analysis
# ------------------------------------------------------------

sales_by_customer = (
    final_sales_analytics
    .groupby("customer_id", as_index=False)
    .agg(
        total_sales=("sales_amount", "sum"),
        total_quantity=("quantity", "sum"),
        order_count=("order_id", "nunique")
    )
    .sort_values(
        "total_sales",
        ascending=False
    )

)

print("\n--- TOP 10 CUSTOMERS ---")
display(sales_by_customer.head(10))


# ------------------------------------------------------------
# Yearly Analysis
# ------------------------------------------------------------

sales_by_year = (
    final_sales_analytics
    .groupby("sales_year", as_index=False)
    .agg(
        total_sales=("sales_amount", "sum"),
        total_quantity=("quantity", "sum"),
        order_count=("order_id", "nunique")
    )
)

print("\n--- YEARLY SALES ---")
display(sales_by_year)

In [ ]:
# ============================================================
# PHASE 10.1 — MONTHLY SALES ANALYSIS
# ============================================================

sales_by_month = (
    final_sales_analytics
    .dropna(subset=["order_date"])
    .groupby(
        final_sales_analytics["order_date"]
        .dt.to_period("M")
    )["sales_amount"]
    .sum()
    .reset_index()
)

sales_by_month.columns = [
    "month",
    "total_sales"
]

sales_by_month["month"] = (
    sales_by_month["month"].astype(str)
)

print("--- MONTHLY SALES ---")
display(sales_by_month)

In [ ]:
# ============================================================
# PHASE 11 — VISUALIZATION
# ============================================================

import matplotlib.pyplot as plt

print("=" * 60)
print("PHASE 11 — VISUALIZATION")
print("=" * 60)


# ------------------------------------------------------------
# 1. Monthly Sales Trend
# ------------------------------------------------------------

plt.figure(figsize=(12, 5))

plt.plot(
    sales_by_month["month"],
    sales_by_month["total_sales"],
    marker="o"
)

plt.title("Monthly Sales Trend")
plt.xlabel("Month")
plt.ylabel("Total Sales")

plt.xticks(rotation=45)

plt.tight_layout()
plt.show()


# ------------------------------------------------------------
# 2. Top Products
# ------------------------------------------------------------

top_products = sales_by_product.head(10)

plt.figure(figsize=(10, 6))

plt.barh(
    top_products["product_id"].astype(str),
    top_products["total_sales"]
)

plt.title("Top 10 Products by Sales")
plt.xlabel("Total Sales")
plt.ylabel("Product ID")

plt.gca().invert_yaxis()

plt.tight_layout()
plt.show()


# ------------------------------------------------------------
# 3. Top Customers
# ------------------------------------------------------------

top_customers = sales_by_customer.head(10)

plt.figure(figsize=(10, 6))

plt.barh(
    top_customers["customer_id"].astype(str),
    top_customers["total_sales"]
)

plt.title("Top 10 Customers by Sales")
plt.xlabel("Total Sales")
plt.ylabel("Customer ID")

plt.gca().invert_yaxis()

plt.tight_layout()
plt.show()


# ------------------------------------------------------------
# 4. Sales Distribution
# ------------------------------------------------------------

plt.figure(figsize=(10, 5))

plt.hist(
    final_sales_analytics["sales_amount"]
    .dropna(),
    bins=30
)

plt.title("Sales Amount Distribution")
plt.xlabel("Sales Amount")
plt.ylabel("Frequency")

plt.tight_layout()
plt.show()


# ------------------------------------------------------------
# 5. Quantity Distribution
# ------------------------------------------------------------

plt.figure(figsize=(10, 5))

plt.hist(
    final_sales_analytics["quantity"]
    .dropna(),
    bins=20
)

plt.title("Quantity Distribution")
plt.xlabel("Quantity")
plt.ylabel("Frequency")

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# PHASE 12 — LOAD
# ============================================================

import os

print("=" * 60)
print("PHASE 12 — LOAD")
print("=" * 60)

output_dir = "/content/etl_output"

os.makedirs(
    output_dir,
    exist_ok=True
)


# ------------------------------------------------------------
# Save Cleaned Data
# ------------------------------------------------------------

sales_integrated.to_csv(
    f"{output_dir}/cleaned_sales.csv",
    index=False
)

customers_lookup.to_csv(
    f"{output_dir}/cleaned_customers.csv",
    index=False
)

products_lookup.to_csv(
    f"{output_dir}/cleaned_products.csv",
    index=False
)


# ------------------------------------------------------------
# Save Dimensions
# ------------------------------------------------------------

dim_customer.to_csv(
    f"{output_dir}/dim_customer.csv",
    index=False
)

dim_product.to_csv(
    f"{output_dir}/dim_product.csv",
    index=False
)

dim_date.reset_index().to_csv(
    f"{output_dir}/dim_date.csv",
    index=False
)


# ------------------------------------------------------------
# Save Fact
# ------------------------------------------------------------

fact_sales.to_csv(
    f"{output_dir}/fact_sales.csv",
    index=False
)


# ------------------------------------------------------------
# Save Analytics
# ------------------------------------------------------------

final_sales_analytics.to_csv(
    f"{output_dir}/final_sales_analytics.csv",
    index=False
)


print("\nFiles successfully saved:")

for file_name in sorted(
    os.listdir(output_dir)
):
    print("✓", file_name)

print(f"\nOutput directory: {output_dir}")

In [ ]:
# ============================================================
# PHASE 13 — ETL VALIDATION
# ============================================================

print("=" * 60)
print("PHASE 13 — ETL VALIDATION")
print("=" * 60)

validation_results = []


# ------------------------------------------------------------
# Sales Row Count
# ------------------------------------------------------------

validation_results.append({
    "check": "Sales row count",
    "value": len(sales_integrated),
    "status": "PASS"
})


# ------------------------------------------------------------
# Fact Row Count
# ------------------------------------------------------------

validation_results.append({
    "check": "Fact row count",
    "value": len(fact_sales),
    "status": (
        "PASS"
        if len(fact_sales) == len(sales_integrated)
        else "FAIL"
    )
})


# ------------------------------------------------------------
# Customer FK
# ------------------------------------------------------------

customer_missing = int(
    fact_sales["customer_key"].isna().sum()
)

validation_results.append({
    "check": "Customer foreign keys",
    "value": customer_missing,
    "status": (
        "PASS"
        if customer_missing == 0
        else "FAIL"
    )
})


# ------------------------------------------------------------
# Product FK
# ------------------------------------------------------------

product_missing = int(
    fact_sales["product_key"].isna().sum()
)

validation_results.append({
    "check": "Product foreign keys",
    "value": product_missing,
    "status": (
        "PASS"
        if product_missing == 0
        else "FAIL"
    )
})


# ------------------------------------------------------------
# Date FK
# ------------------------------------------------------------

date_missing = int(
    fact_sales["date_key"].isna().sum()
)

validation_results.append({
    "check": "Date foreign keys",
    "value": date_missing,
    "status": (
        "PASS"
        if date_missing == 0
        else "FAIL"
    )
})


# ------------------------------------------------------------
# Sales Amount
# ------------------------------------------------------------

sales_missing = int(
    final_sales_analytics[
        "sales_amount"
    ].isna().sum()
)

validation_results.append({
    "check": "Sales amount",
    "value": sales_missing,
    "status": (
        "PASS"
        if sales_missing == 0
        else "WARNING"
    )
})


# ------------------------------------------------------------
# Create Report
# ------------------------------------------------------------

etl_validation_report = pd.DataFrame(
    validation_results
)

display(etl_validation_report)


# ------------------------------------------------------------
# Overall Status
# ------------------------------------------------------------

if (
    etl_validation_report["status"]
    == "FAIL"
).any():

    print("\n❌ ETL VALIDATION FAILED")

elif (
    etl_validation_report["status"]
    == "WARNING"
).any():

    print("\n⚠️ ETL VALIDATION PASSED WITH WARNINGS")

else:

    print("\n✅ ETL VALIDATION PASSED")

In [ ]:
# ============================================================
# PHASE 14 — FINAL ETL SUMMARY
# ============================================================

print("\n" + "=" * 70)
print("FINAL ETL PIPELINE SUMMARY")
print("=" * 70)

print("\nEXTRACT")
print(
    f"  Sales records: "
    f"{len(sales_transformed_df):,}"
)

print("\nTRANSFORM")
print(
    f"  Transformed sales: "
    f"{len(sales_integrated):,}"
)

print("\nPRODUCT INTEGRATION")
print(
    f"  Products: "
    f"{len(dim_product):,}"
)

print(
    f"  Product match rate: "
    f"{match_percentage:.2f}%"
)

print("\nCUSTOMER INTEGRATION")
print(
    f"  Customers: "
    f"{len(dim_customer):,}"
)

print("\nSTAR SCHEMA")
print(
    f"  dim_customer: "
    f"{len(dim_customer):,}"
)

print(
    f"  dim_product: "
    f"{len(dim_product):,}"
)

print(
    f"  dim_date: "
    f"{len(dim_date):,}"
)

print(
    f"  fact_sales: "
    f"{len(fact_sales):,}"
)

print("\nBUSINESS METRICS")
print(
    f"  Total Sales: "
    f"${total_sales:,.2f}"
)

print(
    f"  Total Quantity: "
    f"{total_quantity:,.0f}"
)

print(
    f"  Total Orders: "
    f"{total_orders:,}"
)

print(
    f"  Unique Customers: "
    f"{unique_customers:,}"
)

print(
    f"  Unique Products: "
    f"{unique_products:,}"
)

print(
    f"  Average Order Value: "
    f"${average_order_value:,.2f}"
)

print("\nOUTPUT")
print(
    f"  {output_dir}"
)

print("\nVALIDATION")

if (
    etl_validation_report["status"]
    == "FAIL"
).any():

    print("  ❌ FAILED")

elif (
    etl_validation_report["status"]
    == "WARNING"
).any():

    print("  ⚠️ PASSED WITH WARNINGS")

else:

    print("  ✅ PASSED")


print("\n" + "=" * 70)
print("ETL PIPELINE COMPLETED")
print("=" * 70)

## PHASE 15 — OLAP Cube Construction and Analysis

Now that we have our Star Schema (`fact_sales`, `dim_customer`, `dim_product`, `dim_date`), we can construct an OLAP cube for multi-dimensional analysis.

An OLAP Cube is a powerful tool for business intelligence, allowing users to quickly aggregate and explore data from various perspectives (dimensions) and measure key performance indicators (measures).

### Dimensions:
*   **Date:** `year`, `month_name`, `day`, `quarter` from `dim_date`
*   **Product:** `category`, `brand` from `dim_product`
*   **Customer:** `city`, `country` from `dim_customer`

### Measures:
*   `Total Sales` (`sum` of `sales_amount`)
*   `Total Quantity` (`sum` of `quantity`)
*   `Average Unit Price` (`mean` of `unit_price`)
*   `Order Count` (`nunique` of `order_id`)

We will join the fact table with the dimension tables to create a single analytical dataset, and then use `pandas.pivot_table` to simulate the OLAP cube.

In [ ]:
print('=' * 60)
print('PHASE 15 — OLAP CUBE CONSTRUCTION')
print('=' * 60)

# Ensure all necessary dataframes are available
if 'fact_sales' not in globals():
    raise NameError("fact_sales not found. Please run previous phases.")
if 'dim_customer' not in globals():
    raise NameError("dim_customer not found. Please run previous phases.")
if 'dim_product' not in globals():
    raise NameError("dim_product not found. Please run previous phases.")
if 'dim_date' not in globals():
    raise NameError("dim_date not found. Please run previous phases.")

# --- 1. Merge Fact with Dimensions to create a denormalized view ---

# Merge with dim_customer
olap_df = fact_sales.merge(
    dim_customer[['customer_key', 'first_name', 'last_name', 'city', 'country']],
    on='customer_key',
    how='left'
)

# Merge with dim_product
olap_df = olap_df.merge(
    dim_product[['product_key', 'name', 'brand', 'category']],
    on='product_key',
    how='left'
)

# Merge with dim_date (note: dim_date uses 'date_key' as index, so reset index first if it's an index)
date_lookup = dim_date.copy()
if date_lookup.index.name == 'date_id':
    date_lookup = date_lookup.reset_index()

olap_df = olap_df.merge(
    date_lookup[['date_id', 'year', 'month_name', 'quarter', 'day_of_month']],
    left_on='date_key',
    right_on='date_id',
    how='left'
)

# Drop redundant date_id column from the merge
olap_df = olap_df.drop(columns=['date_id'])

print(f"OLAP-ready DataFrame shape: {olap_df.shape}")
display(olap_df.head())

print("\n--- Building the OLAP Cube (using pivot_table) ---")

# Define the dimensions and measures for the OLAP Cube
dimensions = ['year', 'month_name', 'category', 'brand', 'country', 'city']
measures = {
    'sales_amount': 'sum',
    'quantity': 'sum',
    'unit_price': 'mean'
}

# Create the OLAP cube using pivot_table
olap_cube = pd.pivot_table(
    olap_df,
    values=list(measures.keys()),
    index=['year', 'month_name', 'quarter'], # Rows: e.g., Time dimensions
    columns=['category', 'brand'],             # Columns: e.g., Product dimensions
    aggfunc=measures,
    fill_value=0
)

print("OLAP Cube Head (Multi-level columns and index):")
display(olap_cube.head())

print("\n--- Cube information ---")
print(f"Cube shape: {olap_cube.shape}")
print(f"Cube dimensions (from index): {olap_cube.index.names}")
print(f"Cube dimensions (from columns): {olap_cube.columns.names}")

### OLAP Cube Operations: Slice and Dice

An OLAP cube allows for flexible analysis. Here are examples of common operations:

*   **Slice:** Selecting a single dimension (or a subset of values from a dimension) to view a specific cross-section of the data.
*   **Dice:** Selecting a specific range of values across multiple dimensions, resulting in a sub-cube.
*   **Roll-up:** Aggregating data along a dimension (e.g., from day to month, or product to category).
*   **Drill-down:** Going into more detail along a dimension (e.g., from year to month to day).

Let's demonstrate some of these operations.

In [ ]:
print('=' * 60)
print('OLAP CUBE OPERATIONS: SLICE AND DICE')
print('=' * 60)

# --- 1. Slice: Total Sales for a Specific Year (e.g., 2024) ---
print("\n--- Slice: Total Sales for 2024 ---")
sales_2024 = olap_cube.loc[2024]
display(sales_2024.head())
print(f"Total Sales in 2024: ${sales_2024.loc[:, ('sales_amount', slice(None), slice(None))].sum().sum():,.2f}")


# --- 2. Dice: Sales of 'Electronics' products in 2024 ---
print("\n--- Dice: Sales of 'Electronics' products in 2024 ---")
electronics_2024_sales = olap_cube.loc[2024, (slice(None), 'Electronics', slice(None))]
display(electronics_2024_sales.head())
print(f"Total Electronics Sales in 2024: ${electronics_2024_sales.loc[:, ('sales_amount', slice(None), slice(None))].sum().sum():,.2f}")


# --- 3. Roll-up: Total Sales by Year (from month_name) ---
print("\n--- Roll-up: Total Sales by Year ---")
sales_by_year_rollup = olap_cube.groupby(level='year', axis=0).agg(measures)
display(sales_by_year_rollup)


# --- 4. Drill-down: Sales by Month for a specific category and brand ---
print("\n--- Drill-down: Sales of 'Electronics' 'BrandX' by Month in 2024 ---")
drill_down_data = olap_cube.loc[2024, ('sales_amount', 'Electronics', 'BrandA')].unstack(level='month_name')
display(drill_down_data)


# --- 5. Another Slice: Total quantity sold per product category across all years ---
print("\n--- Slice: Total Quantity by Product Category (All Years) ---")
quantity_by_category = olap_cube.groupby(level='category', axis=1)['quantity'].sum()
display(quantity_by_category.head())

print("\nOLAP Cube operations demonstrated.")

In [ ]:
import pandas as pd
import os

# Define the output directory where files were saved
OUTPUT_DIR = "/content/etl_output"

print("Reloading fact and dimension tables from ETL output directory...")

try:
    fact_sales = pd.read_csv(os.path.join(OUTPUT_DIR, 'fact_sales.csv'))
    dim_customer = pd.read_csv(os.path.join(OUTPUT_DIR, 'dim_customer.csv'))
    dim_product = pd.read_csv(os.path.join(OUTPUT_DIR, 'dim_product.csv'))

    # dim_date might have date_id as index or column, so handle accordingly
    dim_date = pd.read_csv(os.path.join(OUTPUT_DIR, 'dim_date.csv'))
    if 'date_id' in dim_date.columns:
        dim_date = dim_date.set_index('date_id')

    print("Fact and dimension tables reloaded successfully.")
    print(f"fact_sales shape: {fact_sales.shape}")
    print(f"dim_customer shape: {dim_customer.shape}")
    print(f"dim_product shape: {dim_product.shape}")
    print(f"dim_date shape: {dim_date.shape}")

except FileNotFoundError as e:
    print(f"Error reloading files: {e}. Make sure the previous ETL Load phase (Phase 12) was executed successfully.")
    # Re-raise to stop execution if critical files are missing
    raise


### Upload Dataset Folder

To upload your dataset folder, run the following cell. A file selection dialog will appear, allowing you to choose files from your local machine. If you want to upload a folder, you'll need to zip it first and then upload the `.zip` file.

In [9]:
from google.colab import files
import os
import zipfile

print("Please select your dataset files or a zipped folder to upload:")

uploaded = files.upload()

for fn in uploaded.keys():
  print(f'User uploaded file "{fn}" with length {len(uploaded[fn])} bytes')
  if fn.endswith('.zip'):
    with zipfile.ZipFile(fn, 'r') as zip_ref:
      zip_ref.extractall('.')
    print(f'Extracted {fn}')
    os.remove(fn) # Clean up the zip file after extraction

Please select your dataset files or a zipped folder to upload:


Saving customers.csv to customers.csv
Saving products.csv to products.csv
Saving sales.csv to sales.csv
User uploaded file "customers.csv" with length 153373 bytes
User uploaded file "products.csv" with length 157245 bytes
User uploaded file "sales.csv" with length 27830 bytes


### PHASE 1 — DATA EXTRACTION

Now that the files are uploaded, let's load them into pandas DataFrames.

In [10]:
import pandas as pd

def extract_data(file_path):
    """
    Extracts data from a CSV file into a pandas DataFrame.
    """
    try:
        df = pd.read_csv(file_path)
        print(f"Successfully extracted {len(df)} rows from {file_path}")
        return df
    except FileNotFoundError:
        print(f"Error: File not found at {file_path}")
        return pd.DataFrame()
    except Exception as e:
        print(f"An error occurred while extracting {file_path}: {e}")
        return pd.DataFrame()

# Define file paths
sales_file = 'sales.csv'
customers_file = 'customers.csv'
products_file = 'products.csv'

# Extract data
sales_extracted_df = extract_data(sales_file)
customers_extracted_df = extract_data(customers_file)
products_extracted_df = extract_data(products_file)

# Display head of each DataFrame
print("\n--- Sales Data Head ---")
display(sales_extracted_df.head())

print("\n--- Customers Data Head ---")
display(customers_extracted_df.head())

print("\n--- Products Data Head ---")
display(products_extracted_df.head())

Successfully extracted 1000 rows from sales.csv
Successfully extracted 1000 rows from customers.csv
Successfully extracted 1000 rows from products.csv

--- Sales Data Head ---


,OrderID,OrderDate,CustomerID,ProductID,Quantity,UnitPrice,SalesAmount
0,1001,12/16/2024,711,56,4,NaN,NaN
1,1002,2/18/2023,74,987,2,NaN,NaN
2,1003,2/23/2024,283,286,9,NaN,NaN
3,1004,3/27/2024,759,41,2,NaN,NaN
4,1005,8/7/2024,208,884,10,NaN,NaN



--- Customers Data Head ---


,CustomerID,FirstName,LastName,Company,City,Country,Phone1,Phone2,Email,Subscription Date,Website
0,1,Andrew,Goodman,Stewart-Flynn,Rowlandberg,Macao,846-790-4623x4715,(422)787-2331x71127,marieyates@gomez-spencer.info,7/26/2021,http://www.shea.biz/
1,2,Alvin,Lane,"Terry, Proctor and Lawrence",Bethside,Papua New Guinea,124-597-8652x05682,321.441.0588x6218,alexandra86@mccoy.com,6/24/2021,http://www.pena-cole.com/
2,3,Jenna,Harding,Bailey Group,Moniquemouth,China,(335)987-3085x3780,001-680-204-8312,justincurtis@pierce.org,4/5/2020,http://www.booth-reese.biz/
3,4,Fernando,Ford,Moss-Maxwell,Leeborough,Macao,(047)752-3122,048.779.5035x9122,adeleon@hubbard.org,11/29/2020,http://www.hebert.com/
4,5,Kara,Woods,Mccarthy-Kelley,Port Jacksonland,Nepal,+1-360-693-4419x19272,163-627-2565,jesus90@roberson.info,4/22/2022,http://merritt.com/



--- Products Data Head ---


,ProductID,Name,Description,Brand,Category,Price,Currency,Stock,EAN,Color,Size,Availability,InternalID
0,1,Thermostat Drone Heater,Consumer approach woman us those star.,Bradford-Yu,Kitchen Appliances,74,USD,139,8.619790e+12,Orchid,Medium,backorder,38
1,2,Ultra Speaker Iron Grill Advanced One,Point suggest easy money operation could white.,Douglas Group,Fitness Equipment,510,USD,351,3.057220e+12,MediumSeaGreen,Small,backorder,27
2,3,Watch Headphones Kettle,Reach husband education.,Landry-Austin,Beauty & Personal Care,254,USD,409,9.825250e+12,Lavender,XXL,pre_order,2
3,4,Portable Toaster Clock Monitor Silent,Always choose school poor table main.,"Odom, Norton and Foster",Makeup,69,USD,119,9.490810e+12,Pink,XL,in_stock,94
4,5,Pro Toaster Oven,Worry put discuss easy back too career.,"Fowler, Mendoza and Mcdaniel",Automotive,525,USD,727,4.726650e+12,LightCoral,XL,in_stock,95
